<a href="https://colab.research.google.com/github/dhinnn/WDactivity/blob/main/CarFaultDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yt-dlp pydub librosa matplotlib scikit-learn tensorflow noisereduce opencv-python-headless soundfile joblib

import os, glob, random, yt_dlp, cv2, joblib
import numpy as np, matplotlib.pyplot as plt
import librosa, librosa.display, soundfile as sf
import noisereduce as nr
from pydub import AudioSegment
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Set random seed for reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


In [ ]:
fault_links = {
    "Normal": ["https://youtu.be/U5M0gb0nXiw?si=d-bFDo1JfArhLEit"],
    "Misfire": ["https://youtu.be/74ZTtGh5MN8?si=bhVFJYBrWn8V-WyK"],
    "Knocking": ["https://youtu.be/kBWXxWD7g30?si=Km5WH0ecUOmMGr7S"],
    "Tapping_Clicking": ["https://youtu.be/O__pRR2HCS0?si=YwQPcO5ygQYZSidR"],
    "Rod_Knock": ["https://youtu.be/wmtBqNnnvrs?si=5C8CCy6Eh1Z9wvCT"],
    "Exhaust_Leak": ["https://youtu.be/B9vCVByesGI"],
    "Timing_Chain": ["https://youtu.be/2UcWsCP42t0"]
}

dataset_dir = "/content/engine_sounds"
spec_dir = "/content/spectrograms"
os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(spec_dir, exist_ok=True)


In [ ]:
def download_audio(url, label):
    label_dir = os.path.join(dataset_dir, label)
    os.makedirs(label_dir, exist_ok=True)

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': f'{label_dir}/%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'quiet': True
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

for label, urls in fault_links.items():
    for url in urls:
        print(f"⬇️ Downloading {label} from {url}")
        download_audio(url, label)

print("✅ Download Complete!")

# Quick check
for L in fault_links:
    print(L, len(glob.glob(os.path.join(dataset_dir, L, "*.wav"))))


In [ ]:
from scipy.signal import butter, lfilter

def butter_bandpass(lowcut, highcut, fs, order=5):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    return butter(order, [low, high], btype='band')

def bandpass_filter(data, lowcut=50.0, highcut=5000.0, fs=16000, order=5):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    return lfilter(b, a, data)

def denoise_audio(file_path, out_path):
    y, sr = librosa.load(file_path, sr=16000)
    y = bandpass_filter(y, lowcut=50, highcut=5000, fs=sr)
    reduced = nr.reduce_noise(y=y, sr=sr, stationary=True)
    sf.write(out_path, reduced, sr)

def slice_and_denoise(label_dir):
    for file in os.listdir(label_dir):
        if file.endswith(".wav"):
            filepath = os.path.join(label_dir, file)
            clean_path = filepath.replace(".wav", "_clean.wav")
            denoise_audio(filepath, clean_path)

            audio = AudioSegment.from_wav(clean_path)
            duration = len(audio) // 1000
            for i in range(0, duration, 5):
                clip_path = f"{label_dir}/{file[:-4]}_{i}_slice.wav"
                clip = audio[i*1000:(i+5)*1000]
                clip.export(clip_path, format="wav")

for label in fault_links.keys():
    slice_and_denoise(os.path.join(dataset_dir, label))

print("✅ Audio cleaned + sliced into 5-sec clips!")

# Check balance
for L in fault_links:
    print(L, len(glob.glob(os.path.join(dataset_dir, L, "*_slice.wav"))))


In [ ]:
import cv2

def save_spectrogram(file_path, label, count):
    y, sr = librosa.load(file_path, sr=16000)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)

    plt.figure(figsize=(2.24, 2.24), dpi=100)
    librosa.display.specshow(S_dB, sr=sr, x_axis='time', y_axis='mel', cmap='magma')
    plt.axis("off")

    label_dir = os.path.join(spec_dir, label)
    os.makedirs(label_dir, exist_ok=True)
    save_path = os.path.join(label_dir, f"{label}_{count}.png")
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
    plt.close()

    img = cv2.imread(save_path)
    img_resized = cv2.resize(img, (224, 224))
    cv2.imwrite(save_path, img_resized)

count = 0
for label in fault_links.keys():
    label_path = os.path.join(dataset_dir, label)
    for file in os.listdir(label_path):
        if file.endswith("_slice.wav"):
            save_spectrogram(os.path.join(label_path, file), label, count)
            count += 1

print("✅ All spectrograms saved in 224x224 format!")


In [ ]:
import matplotlib.image as mpimg

def save_cropped_spectrogram(img_path, label):
    img = mpimg.imread(img_path)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(img); ax.axis("off")
    plt.tight_layout(pad=0)
    plt.savefig(f"{label}_example.png", dpi=300, bbox_inches='tight', pad_inches=0)
    plt.close()

    fig, ax = plt.subplots(figsize=(4, 5))
    ax.imshow(img); ax.axis("off")
    plt.title(f"Figure: {label.replace('_',' ')} Engine Fault Spectrogram", fontsize=10, pad=10)
    plt.tight_layout()
    plt.savefig(f"{label}_captioned.png", dpi=300, bbox_inches='tight')
    plt.close()

plt.figure(figsize=(15, 8))
for idx, label in enumerate(fault_links.keys()):
    label_dir = os.path.join(spec_dir, label)
    sample_file = random.choice(os.listdir(label_dir))
    sample_path = os.path.join(label_dir, sample_file)
    img = mpimg.imread(sample_path)
    plt.subplot(2, 4, idx+1)
    plt.imshow(img); plt.title(label.replace("_", " ")); plt.axis("off")
    save_cropped_spectrogram(sample_path, label)

plt.tight_layout()
plt.savefig("sample_spectrograms.png", dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
from collections import Counter
files_per_class = {label: len(os.listdir(os.path.join(spec_dir, label))) for label in fault_links}
print("Class counts:", files_per_class)

max_count = max(files_per_class.values())

for label, count in files_per_class.items():
    if count < max_count:
        label_dir = os.path.join(spec_dir, label)
        files = os.listdir(label_dir)
        while len(os.listdir(label_dir)) < max_count:
            f = random.choice(files)
            src = os.path.join(label_dir, f)
            dst = os.path.join(label_dir, f"dup_{len(os.listdir(label_dir))}.png")
            cv2.imwrite(dst, cv2.imread(src))

print("✅ Oversampling complete! New counts:")
for label in fault_links:
    print(label, len(os.listdir(os.path.join(spec_dir, label))))


In [ ]:
# Spectrograms (CNN-LSTM)
X_spec, y_spec = [], []
for label_idx, label in enumerate(fault_links):
    label_dir = os.path.join(spec_dir, label)
    for file in os.listdir(label_dir):
        img = cv2.imread(os.path.join(label_dir, file), cv2.IMREAD_GRAYSCALE)
        X_spec.append(img)
        y_spec.append(label_idx)

X_spec = np.array(X_spec).reshape(-1, 224, 224, 1) / 255.0
y_spec = np.array(y_spec)

X_train_spec, X_test_spec, y_train_spec, y_test_spec = train_test_split(X_spec, y_spec, test_size=0.2, random_state=SEED, stratify=y_spec)

print("CNN-LSTM dataset:", X_train_spec.shape, X_test_spec.shape)

# MFCCs (Random Forest)
X_mfcc, y_mfcc = [], []
for label, urls in fault_links.items():
    label_dir = os.path.join(dataset_dir, label)
    for file in os.listdir(label_dir):
        if file.endswith("_slice.wav"):
            y, sr = librosa.load(os.path.join(label_dir, file), sr=16000)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
            mfcc_mean = np.mean(mfcc, axis=1)
            X_mfcc.append(mfcc_mean)
            y_mfcc.append(label)

X_mfcc = np.array(X_mfcc); y_mfcc = np.array(y_mfcc)
X_train_mfcc, X_test_mfcc, y_train_mfcc, y_test_mfcc = train_test_split(X_mfcc, y_mfcc, test_size=0.2, random_state=SEED, stratify=y_mfcc)

print("RF dataset:", X_train_mfcc.shape, X_test_mfcc.shape)


In [ ]:
# STEP 9: CNN + LSTM Hybrid Model (Fixed)
model = tf.keras.Sequential([
    # CNN feature extractor
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,1)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Flatten spatial features and prepare for LSTM
    layers.Reshape((-1, 128)),   # dynamically infer time steps

    # LSTM over feature sequence
    layers.LSTM(128, return_sequences=False),
    layers.Dropout(0.3),

    # Dense classification head
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(np.unique(y)), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()



In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train_mfcc, y_train_mfcc)

joblib.dump(rf, "/content/rf_model.joblib")


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# --- CNN-LSTM Confusion Matrix ---
if isinstance(y_test_spec[0], str):
    # Map if labels are still strings
    label_to_idx = {label: idx for idx, label in enumerate(fault_links.keys())}
    y_test_spec_idx = np.array([label_to_idx[l] for l in y_test_spec])
else:
    y_test_spec_idx = np.array(y_test_spec)  # Already integer encoded

y_pred_cnnlstm = np.argmax(model.predict(X_test_spec), axis=1)

cm_cnn = confusion_matrix(
    y_test_spec_idx, y_pred_cnnlstm,
    labels=range(len(fault_links))
)
disp_cnn = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=list(fault_links.keys()))
disp_cnn.plot(cmap="Blues", xticks_rotation=45)
plt.title("CNN-LSTM Confusion Matrix")
plt.savefig("cnn_lstm_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ CNN-LSTM Classification Report:")
print(classification_report(
    y_test_spec_idx,
    y_pred_cnnlstm,
    labels=range(len(fault_links)),            # ✅ force only 7 classes
    target_names=list(fault_links.keys()),     # ✅ match names
    zero_division=0
))


# --- Random Forest Confusion Matrix ---
# Make predictions first
y_pred_rf = rf.predict(X_test_mfcc)

# Encode RF test labels + predictions into consistent indices
label_to_idx = {label: idx for idx, label in enumerate(fault_links.keys())}
y_test_rf_idx = np.array([label_to_idx[l] for l in y_test_mfcc])
y_pred_rf_idx = np.array([label_to_idx[l] for l in y_pred_rf])

cm_rf = confusion_matrix(
    y_test_rf_idx, y_pred_rf_idx,
    labels=range(len(fault_links))
)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=list(fault_links.keys()))
disp_rf.plot(cmap="Oranges", xticks_rotation=45)
plt.title("Random Forest Confusion Matrix")
plt.savefig("rf_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Random Forest Classification Report:")
print(classification_report(
    y_test_rf_idx,
    y_pred_rf_idx,
    labels=range(len(fault_links)),           # ✅ force only 7 classes
    target_names=list(fault_links.keys()),    # ✅ match names
    zero_division=0
))


In [ ]:
# ==========================
# Training Curves (CNN-LSTM)
# ==========================
if "history" in locals():
    plt.figure(figsize=(12,5))

    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(history.history.get('accuracy', []), label='Train')
    plt.plot(history.history.get('val_accuracy', []), label='Val')
    plt.legend()
    plt.title("Accuracy")

    # Loss
    plt.subplot(1,2,2)
    plt.plot(history.history.get('loss', []), label='Train')
    plt.plot(history.history.get('val_loss', []), label='Val')
    plt.legend()
    plt.title("Loss")

    plt.tight_layout()
    plt.savefig("cnn_lstm_training_curves.png", dpi=300)
    plt.show()
    print("✅ Training curves plotted and saved.")
else:
    print("⚠️ No training history found. Skipping training curves.")

# ==========================
# Save all results to PDF
# ==========================
from matplotlib.backends.backend_pdf import PdfPages
pdf_path = "engine_fault_evaluation_report.pdf"

with PdfPages(pdf_path) as pdf:
    for img_file, title in [
        ("sample_spectrograms.png", "Sample Spectrograms per Class"),
        ("cnn_lstm_confusion_matrix.png", "CNN-LSTM Confusion Matrix"),
        ("rf_confusion_matrix.png", "Random Forest Confusion Matrix"),
        ("cnn_lstm_training_curves.png", "CNN-LSTM Training Curves"),
    ]:
        if os.path.exists(img_file):
            img = plt.imread(img_file)
            plt.figure(figsize=(12,6))
            plt.imshow(img)
            plt.axis("off")
            plt.title(title)
            pdf.savefig()
            plt.close()

print(f"📑 Final PDF saved as: {pdf_path}")


In [ ]:
# ==========================
# CELL 13: Per-Class Metrics Bar Charts
# ==========================
from sklearn.metrics import precision_recall_fscore_support

# CNN-LSTM metrics
prec_cnn, rec_cnn, f1_cnn, _ = precision_recall_fscore_support(
    y_test_spec_idx, y_pred_cnnlstm, labels=range(len(fault_links)), zero_division=0
)

# RF metrics
prec_rf, rec_rf, f1_rf, _ = precision_recall_fscore_support(
    y_test_rf_idx, y_pred_rf_idx, labels=range(len(fault_links)), zero_division=0
)

labels = list(fault_links.keys())
x = np.arange(len(labels))
bar_width = 0.35

plt.figure(figsize=(16, 5))

# Precision
plt.subplot(1, 3, 1)
plt.bar(x - bar_width/2, prec_cnn, bar_width, label="CNN-LSTM")
plt.bar(x + bar_width/2, prec_rf, bar_width, label="RF")
plt.xticks(x, labels, rotation=45)
plt.ylabel("Precision")
plt.title("Precision per Class")
plt.legend()

# Recall
plt.subplot(1, 3, 2)
plt.bar(x - bar_width/2, rec_cnn, bar_width, label="CNN-LSTM")
plt.bar(x + bar_width/2, rec_rf, bar_width, label="RF")
plt.xticks(x, labels, rotation=45)
plt.ylabel("Recall")
plt.title("Recall per Class")

# F1-score
plt.subplot(1, 3, 3)
plt.bar(x - bar_width/2, f1_cnn, bar_width, label="CNN-LSTM")
plt.bar(x + bar_width/2, f1_rf, bar_width, label="RF")
plt.xticks(x, labels, rotation=45)
plt.ylabel("F1-score")
plt.title("F1-score per Class")

plt.tight_layout()
plt.savefig("cnn_vs_rf_class_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

print("✅ Saved per-class metrics chart as cnn_vs_rf_class_metrics.png")


In [ ]:
# ==========================
# CELL 14.2: Per-Class Noise Robustness Analysis (with Graphs + Debug)
# ==========================
from collections import defaultdict
from sklearn.metrics import accuracy_score

# --- Function to add noise ---
def add_noise_batch(X_specs, noise_level=0.25):
    noisy_specs = []
    for spec in X_specs:
        noise = np.random.normal(0, noise_level, spec.shape)
        noisy_specs.append(spec + noise)
    return np.array(noisy_specs)

# --- Predictions ---
y_pred_clean = np.argmax(model.predict(X_test_spec), axis=1)

X_test_noisy = add_noise_batch(X_test_spec, noise_level=0.25)
y_pred_noisy = np.argmax(model.predict(X_test_noisy), axis=1)

# --- Debug check ---
print("Unique ground-truth labels:", np.unique(y_test_spec_idx))
print("Unique CNN predictions (clean):", np.unique(y_pred_clean))
print("Unique CNN predictions (noisy):", np.unique(y_pred_noisy))

# --- Per-class robustness ---
y_true = y_test_spec_idx
class_accuracies = defaultdict(dict)

for idx, label in enumerate(fault_links.keys()):
    class_idx = np.where(y_true == idx)[0]
    print(f"{label}: found {len(class_idx)} samples")  # 👈 debug line
    if len(class_idx) == 0:
        continue

    acc_clean_cls = accuracy_score(y_true[class_idx], y_pred_clean[class_idx])
    acc_noisy_cls = accuracy_score(y_true[class_idx], y_pred_noisy[class_idx])
    drop = acc_clean_cls - acc_noisy_cls

    class_accuracies[label]["Clean"] = acc_clean_cls
    class_accuracies[label]["Noisy"] = acc_noisy_cls
    class_accuracies[label]["Drop"] = drop

# --- Print results ---
print("\n📊 Per-Class Robustness (Clean vs Noisy):\n")
for label, vals in class_accuracies.items():
    print(f"{label:<15} Clean: {vals['Clean']:.2f} | Noisy: {vals['Noisy']:.2f} | Drop: {vals['Drop']:.2f}")

# --- Visualization: Accuracy ---
labels = list(class_accuracies.keys())
clean_scores = [class_accuracies[l]["Clean"] for l in labels]
noisy_scores = [class_accuracies[l]["Noisy"] for l in labels]
drops = [class_accuracies[l]["Drop"] for l in labels]

x = np.arange(len(labels))
bar_width = 0.35

plt.figure(figsize=(12,6))
plt.bar(x - bar_width/2, clean_scores, bar_width, label="Clean Accuracy")
plt.bar(x + bar_width/2, noisy_scores, bar_width, label="Noisy Accuracy")
plt.xticks(x, labels, rotation=45)
plt.ylim(0, 1.05)
plt.ylabel("Accuracy")
plt.title("Per-Class Accuracy: Clean vs Noisy")
plt.legend()
plt.tight_layout()
plt.savefig("per_class_robustness.png", dpi=300)
plt.show()

# --- Visualization: Accuracy Drop ---
plt.figure(figsize=(10,5))
plt.bar(labels, drops, color="orange")
plt.xticks(rotation=45)
plt.ylabel("Accuracy Drop")
plt.title("Noise Robustness (Accuracy Drop per Class)")
plt.tight_layout()
plt.savefig("per_class_drop.png", dpi=300)
plt.show()


In [ ]:
# ==========================
# CELL 15 (FINAL FIXED): Inference Functions + Demo
# ==========================
import joblib, json

# --- Reload models if needed ---
cnn_model = tf.keras.models.load_model("cnn_lstm_engine_fault.h5")
rf_model = joblib.load("random_forest_engine_fault.pkl")

# --- Load consistent class order ---
if os.path.exists("class_labels.json"):
    with open("class_labels.json", "r") as f:
        class_labels = json.load(f)
else:
    # fallback to fault_links order
    class_labels = list(fault_links.keys())
    # also save for future runs
    with open("class_labels.json", "w") as f:
        json.dump(class_labels, f)

# --- Prediction helper functions ---
def preprocess_for_cnn(file_path):
    """Convert WAV into a spectrogram suitable for CNN-LSTM."""
    y, sr = librosa.load(file_path, sr=16000)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    S_dB = cv2.resize(S_dB, (224,224))   # match training input
    return S_dB.reshape(224,224,1)

def preprocess_for_rf(file_path):
    """Extract MFCC features for Random Forest (MUST match training)."""
    y, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)   # ✅ match training (20 not 40)
    mfcc_mean = np.mean(mfcc.T, axis=0)   # mean-pooling
    return mfcc_mean.reshape(1, -1)

def predict_from_file(file_path):
    """Run both CNN-LSTM and RF models on a new file."""
    # CNN-LSTM
    spec = preprocess_for_cnn(file_path)
    cnn_pred = cnn_model.predict(np.array([spec]))
    cnn_class = int(np.argmax(cnn_pred, axis=1)[0])

    # ✅ Guard against index mismatch
    cnn_label = class_labels[cnn_class] if cnn_class < len(class_labels) else f"Unknown_Class_{cnn_class}"

    # RF
    mfcc_feat = preprocess_for_rf(file_path)
    rf_pred = rf_model.predict(mfcc_feat)[0]

    print("\n🔍 Prediction Results for:", os.path.basename(file_path))
    print(f"CNN-LSTM → {cnn_label} (Prob: {np.max(cnn_pred):.2f})")
    print(f"Random Forest → {rf_pred}")

    return cnn_label, rf_pred


# ==========================
# DEMO: Pick a random test sample
# ==========================
import random
test_label = random.choice(class_labels)
test_dir = os.path.join(dataset_dir, test_label)

# ensure only .wav files
test_files = [f for f in os.listdir(test_dir) if f.endswith(".wav")]
test_file = os.path.join(test_dir, random.choice(test_files))

cnn_out, rf_out = predict_from_file(test_file)


In [ ]:
# ==========================
# CELL 16: Gradio Web App for Demo (FIXED)
# ==========================
import gradio as gr

def classify_engine_fault(file_path):
    """Gradio wrapper: takes an uploaded audio file, runs both models."""
    try:
        # CNN-LSTM
        spec = preprocess_for_cnn(file_path)
        cnn_pred = cnn_model.predict(np.array([spec]))
        cnn_class = int(np.argmax(cnn_pred, axis=1)[0])
        cnn_label = class_labels[cnn_class] if cnn_class < len(class_labels) else f"Unknown_{cnn_class}"

        # RF
        mfcc_feat = preprocess_for_rf(file_path)
        rf_pred = rf_model.predict(mfcc_feat)[0]

        return {
            "CNN-LSTM Prediction": cnn_label,
            "CNN-LSTM Confidence": f"{np.max(cnn_pred):.2f}",
            "Random Forest Prediction": rf_pred
        }
    except Exception as e:
        return {"Error": str(e)}

# Build Gradio interface
demo = gr.Interface(
    fn=classify_engine_fault,
    inputs=gr.Audio(type="filepath", label="Upload Engine Sound (.wav)"),  # ✅ FIXED
    outputs="json",
    title="🔊 Engine Fault Detection",
    description="Upload an engine sound (.wav). The system will predict if it is Normal or Faulty (Misfire, Knocking, Tapping/Clicking, Rod Knock, Exhaust Leak, Timing Chain)."
)

demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bbb292aa65add8f1e2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
